s

In [1]:
import sys
print(sys.executable)


d:\INEI_DESPLIEGUE_WEB\PROYECTO_FINAL\01Proyecto_Clasificacion\.venv\Scripts\python.exe


# Proyecto de Clasificación ENAHO 2024 - NBI

Este notebook está corregido para trabajar con la ruta fija:

`D:\INEI_DESPLIEGUE_WEB\PROYECTO_FINAL\01Proyecto_Clasificacion`

Al ejecutar **Run All**, genera en esa misma carpeta:

- `modelo_enaho_nbi.joblib`
- `app_streamlit.py`
- `requirements.txt`
- `README.md`
- `importancia_variables_enaho.csv`
- `casos_prueba_streamlit_enaho.csv`
- `ejecutar_app_streamlit.bat`

También corrige el error de `matplotlib` para graficar la importancia de variables.


In [ ]:
# Bloque 1: instalar librerías faltantes dentro del kernel actual
import sys
import subprocess
import importlib.util

paquetes = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "streamlit": "streamlit",
    "matplotlib": "matplotlib",
}

for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])

print("Librerías verificadas correctamente.")

In [ ]:
# Bloque 2: importar librerías necesarias para cargar datos, entrenar modelos, evaluar y guardar archivos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from joblib import dump, load

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

print("Importaciones realizadas correctamente.")

In [ ]:
# Bloque 3: definir ruta fija del proyecto y rutas de salida
RUTA_PROYECTO = Path(r"D:\INEI_DESPLIEGUE_WEB\PROYECTO_FINAL\01Proyecto_Clasificacion")

RUTA_APP = RUTA_PROYECTO / "app_streamlit.py"
RUTA_MODELO = RUTA_PROYECTO / "modelo_enaho_nbi.joblib"
RUTA_IMPORTANCIA = RUTA_PROYECTO / "importancia_variables_enaho.csv"
RUTA_CASOS = RUTA_PROYECTO / "casos_prueba_streamlit_enaho.csv"
RUTA_REQUIREMENTS = RUTA_PROYECTO / "requirements.txt"
RUTA_README = RUTA_PROYECTO / "README.md"
RUTA_BAT = RUTA_PROYECTO / "ejecutar_app_streamlit.bat"

if not RUTA_PROYECTO.exists():
    raise FileNotFoundError(f"No existe la carpeta del proyecto: {RUTA_PROYECTO}")

print("Carpeta del proyecto encontrada:")
print(RUTA_PROYECTO)
print("El modelo se guardará en:")
print(RUTA_MODELO)

In [ ]:
# Bloque 4: buscar automáticamente el archivo CSV ENAHO dentro de la carpeta del proyecto
archivos_csv = list(RUTA_PROYECTO.glob("*.csv"))

archivos_excluidos = {
    "importancia_variables_enaho.csv",
    "casos_prueba_streamlit_enaho.csv",
}

candidatos_csv = [
    ruta for ruta in archivos_csv
    if ruta.name not in archivos_excluidos
]

preferidos = [
    ruta for ruta in candidatos_csv
    if "enaho" in ruta.name.lower() or "enaho01" in ruta.name.lower()
]

if len(preferidos) >= 1:
    RUTA_DATASET = preferidos[0]
elif len(candidatos_csv) == 1:
    RUTA_DATASET = candidatos_csv[0]
else:
    nombres = [ruta.name for ruta in candidatos_csv]
    raise FileNotFoundError(
        "No se pudo identificar un único CSV de entrada. "
        f"Archivos CSV encontrados: {nombres}. "
        "Deja solo el CSV ENAHO base o renómbralo con la palabra ENAHO."
    )

print("Dataset seleccionado:")
print(RUTA_DATASET)

In [ ]:
# Bloque 5: leer el dataset ENAHO desde la ruta fija
datos = pd.read_csv(RUTA_DATASET, encoding="utf-8-sig", low_memory=False)

datos.columns = datos.columns.astype(str).str.strip()
datos = datos.replace(r"^\s*$", np.nan, regex=True)

print("Dimensión de la base original:", datos.shape)
display(datos.head())

In [ ]:
# Bloque 6: preparar columnas NBI y crear TARGET_NBI
nbi_cols = ["NBI1", "NBI2", "NBI3", "NBI4", "NBI5"]

faltantes_nbi = [col for col in nbi_cols if col not in datos.columns]

if faltantes_nbi:
    raise ValueError(f"Faltan columnas NBI en el dataset: {faltantes_nbi}")

for col in nbi_cols:
    datos[col] = pd.to_numeric(datos[col], errors="coerce")

datos_modelo = datos.dropna(subset=nbi_cols).copy()
datos_modelo["TARGET_NBI"] = (datos_modelo[nbi_cols].sum(axis=1) > 0).astype(int)

print("Distribución absoluta del target:")
print(datos_modelo["TARGET_NBI"].value_counts())

print("\nDistribución porcentual del target:")
print(datos_modelo["TARGET_NBI"].value_counts(normalize=True))

if datos_modelo["TARGET_NBI"].nunique() < 2:
    raise ValueError("El target tiene una sola clase. No se puede entrenar un clasificador válido.")

In [ ]:
# Bloque 7: definir variables predictoras categóricas, numéricas, features y target
categorical_features = [
    "DOMINIO", "ESTRATO", "P22", "P24A", "P24B",
    "P101", "P102", "P103", "P104", "P110", "P111A",
    "P1121", "P1141",
]

numeric_features = ["P106", "P117T2", "P117T3", "P117T4"]

features = categorical_features + numeric_features
target = "TARGET_NBI"

faltantes_features = [col for col in features + [target] if col not in datos_modelo.columns]

if faltantes_features:
    raise ValueError(f"Faltan estas columnas en la base: {faltantes_features}")

print("Todas las variables necesarias existen correctamente.")
print("Variables categóricas:", categorical_features)
print("Variables numéricas:", numeric_features)

In [ ]:
# Bloque 8: convertir variables a tipos adecuados para evitar errores en el pipeline
for col in categorical_features:
    datos_modelo[col] = datos_modelo[col].apply(lambda x: np.nan if pd.isna(x) else str(x))

for col in numeric_features:
    datos_modelo[col] = pd.to_numeric(datos_modelo[col], errors="coerce")

X = datos_modelo[features].copy()
y = datos_modelo[target].copy()

print("Filas usadas para modelamiento:", len(datos_modelo))
print("Columnas de X:", X.shape[1])

In [ ]:
# Bloque 9: dividir la base en entrenamiento y prueba manteniendo la proporción del target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y,
)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)

In [ ]:
# Bloque 10: crear OneHotEncoder compatible con versiones nuevas y antiguas de scikit-learn
def crear_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# Bloque 11: crear preprocesador para variables numéricas y categóricas
def crear_preprocessor():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", crear_onehot_encoder()),
    ])

    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

print("Preprocesador definido correctamente.")

In [ ]:
# Bloque 12: definir modelos de clasificación para comparar
modelos = {
    "RandomForest": RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=123,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=160,
        learning_rate=0.05,
        max_depth=3,
        random_state=123,
    ),
}

print("Modelos definidos:", list(modelos.keys()))

In [ ]:
# Bloque 13: entrenar modelos, calcular métricas y guardar resultados
resultados = {}
pipelines = {}

for nombre, modelo in modelos.items():
    print(f"Entrenando modelo: {nombre}")

    pipe = Pipeline(steps=[
        ("preprocess", crear_preprocessor()),
        ("model", modelo),
    ])

    pipe.fit(X_train, y_train)

    y_pred_temp = pipe.predict(X_test)
    y_proba_temp = pipe.predict_proba(X_test)[:, 1]

    resultados[nombre] = {
        "accuracy": float(accuracy_score(y_test, y_pred_temp)),
        "precision_1": float(precision_score(y_test, y_pred_temp, zero_division=0)),
        "recall_1": float(recall_score(y_test, y_pred_temp, zero_division=0)),
        "f1_1": float(f1_score(y_test, y_pred_temp, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, y_proba_temp)),
        "confusion_matrix": confusion_matrix(y_test, y_pred_temp).tolist(),
    }

    pipelines[nombre] = pipe

resultados_df = pd.DataFrame(resultados).T
display(resultados_df)

In [ ]:
# Bloque 14: seleccionar el mejor modelo priorizando roc_auc y luego f1 de la clase 1
mejor_modelo_nombre = max(
    resultados,
    key=lambda k: (resultados[k]["roc_auc"], resultados[k]["f1_1"]),
)

mejor_modelo = pipelines[mejor_modelo_nombre]

print("Mejor modelo seleccionado:", mejor_modelo_nombre)

In [ ]:
# Bloque 15: evaluar el mejor modelo con reporte de clasificación
y_pred = mejor_modelo.predict(X_test)
y_proba = mejor_modelo.predict_proba(X_test)[:, 1]

print("Reporte de clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Bloque 16: calcular importancia de variables agrupando variables one-hot por variable original
def obtener_importancia(pipe, numeric_features, categorical_features):
    modelo = pipe.named_steps["model"]
    pre = pipe.named_steps["preprocess"]

    if not hasattr(modelo, "feature_importances_"):
        return pd.DataFrame(columns=["feature", "importance"])

    nombres = []
    nombres.extend(numeric_features)

    encoder_final = pre.named_transformers_["cat"].named_steps["encoder"]
    nombres.extend(list(encoder_final.get_feature_names_out(categorical_features)))

    importancias = modelo.feature_importances_

    raw = pd.DataFrame({
        "processed_feature": nombres,
        "importance": importancias,
    })

    def original_feature(nombre):
        for col in categorical_features:
            if str(nombre).startswith(col + "_"):
                return col
        return nombre

    raw["feature"] = raw["processed_feature"].apply(original_feature)

    importancia_final = (
        raw.groupby("feature", as_index=False)["importance"]
        .sum()
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    return importancia_final


importancia = obtener_importancia(mejor_modelo, numeric_features, categorical_features)

print("Importancia de variables:")
display(importancia)

In [ ]:
# Bloque 17: graficar importancia de variables dentro del notebook
if not importancia.empty:
    ax = importancia.sort_values("importance").plot.barh(
        x="feature",
        y="importance",
        figsize=(8, 6),
        title="Importancia de variables",
        legend=False,
    )

    ax.set_xlabel("Importancia")
    ax.set_ylabel("Variable")
    plt.tight_layout()
    plt.show()
else:
    print("No hay importancia de variables para graficar.")

In [ ]:
# Bloque 18: construir casos de prueba para comparar VS Code y Streamlit
casos = X_test.copy().reset_index(drop=True)

casos["TARGET_REAL"] = y_test.reset_index(drop=True)
casos["PREDICCION_MODELO"] = y_pred
casos["PROBABILIDAD_TARGET_1"] = np.round(y_proba, 6)

casos_0 = casos[casos["PREDICCION_MODELO"] == 0].head(3)
casos_1 = casos[casos["PREDICCION_MODELO"] == 1].head(3)

casos_prueba = pd.concat([casos_0, casos_1], ignore_index=True)

if casos_prueba.empty:
    casos_prueba = casos.head(6).copy()

print("Casos de prueba para Streamlit:")
display(casos_prueba)

In [ ]:
# Bloque 19: crear opciones categóricas, rangos numéricos y etiquetas para Streamlit
categorical_options = {}

for col in categorical_features:
    opciones = X[col].dropna().astype(str).unique().tolist()
    opciones = sorted(opciones)
    if len(opciones) == 0:
        opciones = ["No disponible"]
    categorical_options[col] = opciones

numeric_ranges = {}

for col in numeric_features:
    serie = pd.to_numeric(X[col], errors="coerce")

    if serie.dropna().empty:
        numeric_ranges[col] = {
            "min": 0.0,
            "max": 1.0,
            "median": 0.0,
        }
    else:
        numeric_ranges[col] = {
            "min": float(np.nanmin(serie)),
            "max": float(np.nanmax(serie)),
            "median": float(np.nanmedian(serie)),
        }

feature_labels = {
    "DOMINIO": "Dominio geográfico",
    "ESTRATO": "Estrato",
    "P22": "Tipo de vivienda",
    "P24A": "Material predominante en paredes",
    "P24B": "Material predominante en pisos",
    "P101": "Abastecimiento de agua",
    "P102": "Servicio higiénico",
    "P103": "Alumbrado eléctrico",
    "P104": "Combustible usado para cocinar",
    "P110": "Régimen de tenencia",
    "P111A": "Título de propiedad",
    "P1121": "Equipamiento o servicio del hogar",
    "P1141": "Acceso a tecnologías o servicios",
    "P106": "Número de habitaciones",
    "P117T2": "Ingreso o gasto 2",
    "P117T3": "Ingreso o gasto 3",
    "P117T4": "Ingreso o gasto 4",
}

print("Metadata auxiliar creada correctamente.")

In [ ]:
# Bloque 20: crear metadata compatible con app_streamlit.py
metadata = {
    "dataset": RUTA_DATASET.name,
    "target_name": "TARGET_NBI",
    "target_definition": "1 si el hogar presenta al menos una Necesidad Básica Insatisfecha; 0 si no presenta ninguna.",
    "target_distribution": {str(k): int(v) for k, v in y.value_counts().to_dict().items()},
    "n_rows_modeling": int(len(datos_modelo)),
    "features": features,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "categorical_options": categorical_options,
    "numeric_ranges": numeric_ranges,
    "feature_labels": feature_labels,
    "best_model": mejor_modelo_nombre,
    "metrics": resultados,
    "feature_importance": importancia.to_dict(orient="records"),
    "sample_cases": casos_prueba.to_dict(orient="records"),
}

paquete_modelo = {
    "model": mejor_modelo,
    "metadata": metadata,
}

print("Paquete del modelo creado correctamente.")

In [ ]:
# Bloque 21: guardar modelo_enaho_nbi.joblib y archivos auxiliares en la misma carpeta del proyecto
dump(paquete_modelo, RUTA_MODELO)

importancia.to_csv(
    RUTA_IMPORTANCIA,
    index=False,
    encoding="utf-8-sig",
)

casos_prueba.to_csv(
    RUTA_CASOS,
    index=False,
    encoding="utf-8-sig",
)

print("Modelo guardado en:", RUTA_MODELO)
print("Importancia guardada en:", RUTA_IMPORTANCIA)
print("Casos de prueba guardados en:", RUTA_CASOS)

if not RUTA_MODELO.exists():
    raise FileNotFoundError("No se creó modelo_enaho_nbi.joblib correctamente.")

print("Archivo modelo_enaho_nbi.joblib creado correctamente.")

In [ ]:
# Bloque 22: generar app_streamlit.py corregido para cargar modelo_enaho_nbi.joblib
app_streamlit_code = r'''import streamlit as st
import pandas as pd
from joblib import load
from pathlib import Path

# =========================
# App Streamlit - ENAHO 2024
# Proyecto de clasificación
# =========================

st.set_page_config(
    page_title="Clasificación ENAHO 2024 - NBI",
    page_icon="🏠",
    layout="wide"
)


# Bloque: cargar el modelo entrenado desde la misma carpeta de app_streamlit.py
@st.cache_resource
def cargar_modelo():
    model_path = Path(__file__).resolve().parent / "modelo_enaho_nbi.joblib"

    if not model_path.exists():
        st.error(f"No se encontró el modelo en la ruta: {model_path}")
        st.stop()

    paquete = load(model_path)

    if "model" not in paquete or "metadata" not in paquete:
        st.error("El archivo modelo_enaho_nbi.joblib no contiene las claves 'model' y 'metadata'.")
        st.stop()

    return paquete["model"], paquete["metadata"]


# Bloque: cargar modelo y metadata para usar la app
clf, metadata = cargar_modelo()

st.title("Modelo de Clasificación ENAHO 2024")
st.markdown(
    """
    Esta aplicación predice si un hogar presenta al menos una Necesidad Básica Insatisfecha (NBI).

    **Target:** `TARGET_NBI`  
    **0:** No presenta NBI  
    **1:** Presenta al menos una NBI
    """
)
st.markdown("---")

features = metadata["features"]
cat_features = metadata["categorical_features"]
num_features = metadata["numeric_features"]
cat_options = metadata["categorical_options"]
num_ranges = metadata["numeric_ranges"]
labels = metadata.get("feature_labels", {})

st.sidebar.header("Formulario de predicción")

sample_cases = metadata.get("sample_cases", [])
sample_names = ["Sin caso de prueba"] + [f"Caso {i+1}" for i in range(len(sample_cases))]
selected_sample = st.sidebar.selectbox(
    "Cargar caso de prueba para comparar con VS Code",
    sample_names,
)

default_values = {}
if selected_sample != "Sin caso de prueba":
    idx = sample_names.index(selected_sample) - 1
    default_values = sample_cases[idx]

st.sidebar.markdown("### Variables categóricas")
input_data = {}

for col in cat_features:
    options = [str(x) for x in cat_options[col]]
    default = str(default_values.get(col, options[0]))
    index = options.index(default) if default in options else 0
    input_data[col] = st.sidebar.selectbox(
        f"{labels.get(col, col)}",
        options=options,
        index=index,
    )

st.sidebar.markdown("### Variables numéricas")

for col in num_features:
    r = num_ranges[col]
    min_v = float(r["min"])
    max_v = float(r["max"])
    median_v = float(r["median"])
    default = float(default_values.get(col, median_v))

    if default < min_v:
        default = min_v

    if default > max_v:
        default = max_v

    input_data[col] = st.sidebar.number_input(
        f"{labels.get(col, col)}",
        min_value=min_v,
        max_value=max_v,
        value=default,
        step=1.0,
    )

col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("Datos ingresados")
    obs = pd.DataFrame([input_data], columns=features)
    st.dataframe(obs, use_container_width=True)

    predecir = st.button("Predecir", type="primary")

with col2:
    st.subheader("Información del modelo")
    st.write(f"**Modelo seleccionado:** {metadata['best_model']}")
    st.write(f"**Base usada:** {metadata['dataset']}")
    st.write(f"**Filas usadas para modelamiento:** {metadata.get('n_rows_modeling', 'No disponible')}")
    st.write(f"**Definición del target:** {metadata['target_definition']}")

if predecir:
    pred = int(clf.predict(obs)[0])
    proba_1 = float(clf.predict_proba(obs)[0][1])
    proba_0 = float(clf.predict_proba(obs)[0][0])

    st.markdown("---")
    st.subheader("Resultado de la predicción")

    if pred == 1:
        st.error(f"Clase predicha: 1 - Hogar con al menos una NBI. Probabilidad: {proba_1:.4f}")
    else:
        st.success(f"Clase predicha: 0 - Hogar sin NBI. Probabilidad: {proba_0:.4f}")

    st.write("Probabilidades por clase:")
    st.dataframe(
        pd.DataFrame({
            "Clase": ["0 - Sin NBI", "1 - Con NBI"],
            "Probabilidad": [proba_0, proba_1],
        }),
        use_container_width=True,
    )

    if selected_sample != "Sin caso de prueba":
        real = default_values.get("TARGET_REAL", "No disponible")
        pred_vs = default_values.get("PREDICCION_MODELO", "No disponible")
        prob_vs = default_values.get("PROBABILIDAD_TARGET_1", "No disponible")
        st.info(
            f"Comparación con VS Code para {selected_sample}: "
            f"target real={real}, predicción guardada={pred_vs}, "
            f"probabilidad clase 1={prob_vs}."
        )

st.markdown("---")
st.subheader("Importancia de variables")

importance = pd.DataFrame(metadata["feature_importance"])

if not importance.empty:
    importance_plot = importance.set_index("feature")["importance"].sort_values(ascending=True)
    st.bar_chart(importance_plot)
    st.dataframe(importance, use_container_width=True)
else:
    st.info("No hay importancia de variables disponible para mostrar.")

st.markdown("---")

with st.expander("Auditoría técnica del despliegue"):
    st.markdown(
        """
        - La aplicación carga un único archivo `modelo_enaho_nbi.joblib` que contiene el pipeline completo.
        - El pipeline incluye preprocesamiento de variables numéricas y categóricas.
        - Las variables categóricas se procesan con OneHotEncoder.
        - Las variables numéricas se imputan y escalan.
        - El modelo final fue seleccionado comparando Random Forest y Gradient Boosting.
        - El formulario usa `st.sidebar` con listas desplegables y entradas numéricas.
        - Los casos de prueba permiten comprobar que la predicción web coincide con VS Code.
        """
    )
'''

RUTA_APP.write_text(app_streamlit_code, encoding="utf-8")

print("app_streamlit.py generado/corregido en:", RUTA_APP)

In [ ]:
# Bloque 23: generar requirements.txt actualizado con matplotlib
requirements_text = """streamlit>=1.30
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
joblib>=1.3
matplotlib>=3.7
notebook>=7.0
ipykernel>=6.0
"""

RUTA_REQUIREMENTS.write_text(requirements_text, encoding="utf-8")

print("requirements.txt generado en:", RUTA_REQUIREMENTS)

In [ ]:
# Bloque 24: generar README.md básico del proyecto
readme_text = """# Proyecto de Clasificación ENAHO 2024 - NBI

Este proyecto predice si un hogar presenta al menos una Necesidad Básica Insatisfecha usando datos ENAHO 2024.

## Archivos principales

- `app_streamlit.py`: aplicación web en Streamlit.
- `modelo_enaho_nbi.joblib`: modelo entrenado con pipeline completo.
- `importancia_variables_enaho.csv`: importancia de variables.
- `casos_prueba_streamlit_enaho.csv`: casos para validar predicciones.
- `requirements.txt`: librerías necesarias.
- `ejecutar_app_streamlit.bat`: archivo para ejecutar la app en Windows.

## Ruta del proyecto

```text
D:\\INEI_DESPLIEGUE_WEB\\PROYECTO_FINAL\\01Proyecto_Clasificacion
```

## Ejecutar aplicación

```bash
cd /d "D:\\INEI_DESPLIEGUE_WEB\\PROYECTO_FINAL\\01Proyecto_Clasificacion"
streamlit run app_streamlit.py
```

## Target

- `0`: hogar sin NBI.
- `1`: hogar con al menos una NBI.
"""

RUTA_README.write_text(readme_text, encoding="utf-8")

print("README.md generado en:", RUTA_README)

In [ ]:
# Bloque 25: generar archivo .bat para ejecutar Streamlit con doble clic desde Windows
bat_text = f"""@echo off
cd /d "{RUTA_PROYECTO}"
streamlit run app_streamlit.py
pause
"""

RUTA_BAT.write_text(bat_text, encoding="utf-8")

print("Archivo ejecutar_app_streamlit.bat generado en:", RUTA_BAT)

In [ ]:
# Bloque 26: validar que el modelo se pueda cargar correctamente desde modelo_enaho_nbi.joblib
modelo_cargado = load(RUTA_MODELO)

clf = modelo_cargado["model"]
metadata_cargada = modelo_cargado["metadata"]

print("Modelo cargado correctamente.")
print("Modelo seleccionado:", metadata_cargada["best_model"])
print("Cantidad de features:", len(metadata_cargada["features"]))

In [ ]:
# Bloque 27: probar una predicción con el primer caso de prueba
caso_1 = casos_prueba[features].iloc[[0]]

prediccion = int(clf.predict(caso_1)[0])
probabilidad_clase_1 = float(clf.predict_proba(caso_1)[0][1])

print("Predicción de prueba:", prediccion)
print("Probabilidad clase 1:", round(probabilidad_clase_1, 6))

display(caso_1)

In [ ]:
# Bloque 28: mostrar resumen final de archivos generados
archivos_generados = [
    RUTA_MODELO,
    RUTA_APP,
    RUTA_IMPORTANCIA,
    RUTA_CASOS,
    RUTA_REQUIREMENTS,
    RUTA_README,
    RUTA_BAT,
]

print("Resumen final de archivos generados:")

for ruta in archivos_generados:
    estado = "OK" if ruta.exists() else "FALTA"
    print(f"{estado}: {ruta}")

print("\nPara ejecutar la app:")
print(f'cd /d "{RUTA_PROYECTO}"')
print("streamlit run app_streamlit.py")